# SMM Route Optimizer — Project Report

**Subject:** Supply-chain inspired route optimization over a procedurally generated graph  
**Stack:** Next.js (TypeScript) · Python FastAPI · Dijkstra + TSP (nearest-neighbour + 2-opt)

---

## 1. Project Overview

The goal is to find the optimal routes for one or more vehicles visiting a set of cities on a weighted graph, minimizing one of three objectives: **total time**, **total distance**, or **total fuel consumption**.

The application is structured in three layers:

| Layer | Technology | Role |
|-------|-----------|------|
| Frontend | Next.js 15 / React 19 | Map editor, objectives UI, animation playback |
| JS Solver | TypeScript (browser) | Default solver, runs entirely client-side |
| Python Solver | FastAPI + uvicorn | Optional backend solver, auto-detected via `/health` |

Both solvers implement the same algorithm. The Python solver is activated automatically when the FastAPI server is running on port 8000.

---

## 2. Graph Structure

The map is modeled as an **undirected weighted multigraph** $G = (V, E)$.

### Nodes

Two types of nodes exist:

- **Node** (`type = 'node'`): a city or main location — the only type that can be selected as a destination.
- **Subnode** (`type = 'subnode'`): a road junction, used to give edges a curved or indirect geometry. Invisible to the user as a destination.

### Edges

Each edge connects two nodes (or subnodes) and is composed of one or more **segments**. Each segment carries:

| Field | Type | Description |
|-------|------|-------------|
| `distance` | km | Physical length |
| `speed` | km/h | Speed limit |
| `traffic` | [0, 1] | Traffic load factor |

### Edge total distance

$$d_e = \sum_{s \in e} d_s$$

### Effective speed on a segment

Traffic reduces the effective speed with a capped efficiency factor:

$$\text{eff}_s = \max(0.05,\ 1 - 0.6 \cdot t_s)$$

$$v_s^{\text{eff}} = \min(v_s,\ v^{\text{max}}) \cdot \text{eff}_s$$

where $t_s \in [0,1]$ is the traffic load and $v^{\text{max}}$ is the vehicle speed cap.

### Travel time on a segment (minutes)

$$T_s = \frac{d_s}{v_s^{\text{eff}}} \times 60$$

### Fuel consumption on a segment (litres)

A linear traffic-dependent model:

$$F_s = d_s \cdot (0.07 + 0.05 \cdot t_s)$$

At zero traffic this gives **7 L/100 km**; at full traffic it rises to **12 L/100 km**.

---

## 3. Algorithm Pipeline

```
Input: MapData, Objective, Vehicles
       |
       v
  [1] Build adjacency graph (one per distinct vehicle speed cap)
       |
       v
  [2] Dijkstra — all-pairs shortest paths over the set of cities to visit
       |
       v
  [3] Assign cities to vehicle instances
       |
       v
  [4] TSP — nearest-neighbour construction + 2-opt improvement (per vehicle)
       |
       v
  [5] Expand routes into full node/edge paths
       |
       v
  [6] Compute metrics & check constraints
       |
       v
  Output: frames (animation) + result (metrics)
```

---

## 4. Dijkstra — Shortest Path

Dijkstra's algorithm finds the minimum-cost path from a source node to all other nodes in a non-negative weighted graph.

**Complexity:** $O((V + E) \log V)$ with a binary heap.

### Relaxation condition

For each neighbour $v$ of the current node $u$:

$$\text{cost}[v] > \text{cost}[u] + w(u, v) \implies \text{cost}[v] \leftarrow \text{cost}[u] + w(u, v)$$

The weight $w(u,v)$ is one of $T_e$, $d_e$, or $F_e$ depending on the optimization objective.

The implementation runs **one Dijkstra pass per city** (used as source), producing an all-pairs cost matrix over the cities to visit. It reuses this matrix for TSP.

---

## 5. TSP — Nearest-Neighbour + 2-opt

The Travelling Salesman Problem (TSP) asks: given a set of cities and pairwise costs, find the shortest tour visiting each city exactly once.

TSP is NP-hard. We use a two-phase heuristic:

### Phase 1 — Nearest-neighbour construction

Starting from the departure city (or the city with the smallest average distance to others), greedily pick the nearest unvisited city at each step.

$$\text{next} = \arg\min_{c \notin \text{visited}} \text{cost}(\text{current}, c)$$

**Complexity:** $O(n^2)$

### Phase 2 — 2-opt improvement

Iteratively reverse sub-sequences of the route to reduce total cost. A swap of edges $(i, i+1)$ and $(j, j+1)$ is accepted if:

$$\text{cost}(r_i, r_j) + \text{cost}(r_{i+1}, r_{j+1}) < \text{cost}(r_i, r_{i+1}) + \text{cost}(r_j, r_{j+1})$$

The loop runs up to 20 iterations or until no improvement is found.

**Complexity per iteration:** $O(n^2)$

---

## 6. Code Reference — Key Functions from `scripts/solver.py`

### 6.1 Adjacency graph construction

In [ ]:
import heapq, math
from dataclasses import dataclass, field
from typing import Any

@dataclass
class GEdge:
    id: str
    frm: str
    to: str
    dist: float
    time: float
    fuel: float

# Simplified segment dataclass for the notebook
@dataclass
class Segment:
    distance: float   # km
    speed: float      # km/h
    traffic: float    # 0-1

@dataclass
class Edge:
    id: str
    nodeA: str
    nodeB: str
    segments: list
    totalDistance: float = 0.0

def build_adj(edges: list, nodes: list[str], vehicle_speed_max: float = math.inf) -> dict[str, list]:
    """Build bidirectional adjacency list from edge list."""
    adj: dict[str, list] = {n: [] for n in nodes}
    for e in edges:
        dist = time_ = fuel = 0.0
        for s in e.segments:
            dist   += s.distance
            eff     = max(0.05, 1 - s.traffic * 0.6)                   # traffic efficiency
            v_eff   = min(s.speed, vehicle_speed_max) * eff             # effective speed
            time_  += (s.distance / v_eff) * 60                        # minutes
            fuel   += s.distance * (0.07 + s.traffic * 0.05)           # litres
        for frm, to in [(e.nodeA, e.nodeB), (e.nodeB, e.nodeA)]:
            adj.setdefault(frm, []).append(GEdge(e.id, frm, to, dist, time_, fuel))
    return adj

print("build_adj defined")

### 6.2 Dijkstra

In [ ]:
def dijkstra(adj: dict, source: str, key: str):
    """Return (cost, prev, prev_edge) dicts from source using the given edge key."""
    cost     = {n: math.inf for n in adj}
    prev     = {n: None     for n in adj}
    prev_edge= {n: None     for n in adj}
    cost[source] = 0.0
    pq: list = [(0.0, source)]
    vis: set  = set()
    while pq:
        c, u = heapq.heappop(pq)
        if u in vis:
            continue
        vis.add(u)
        for e in adj.get(u, []):
            val = getattr(e, key)
            nc  = c + val
            if nc < cost.get(e.to, math.inf):
                cost[e.to]      = nc
                prev[e.to]      = u
                prev_edge[e.to] = e.id
                heapq.heappush(pq, (nc, e.to))
    return cost, prev, prev_edge

def get_path(cost, prev, prev_edge, src, tgt):
    """Reconstruct path from prev pointers."""
    if math.isinf(cost.get(tgt, math.inf)):
        return {"nodes": [], "edges": [], "cost": math.inf}
    nodes, edges = [], []
    cur = tgt
    while cur is not None and cur != src:
        nodes.insert(0, cur)
        e = prev_edge.get(cur)
        if e:
            edges.insert(0, e)
        cur = prev.get(cur)
    nodes.insert(0, src)
    return {"nodes": nodes, "edges": edges, "cost": cost[tgt]}

print("dijkstra defined")

### 6.3 TSP — nearest-neighbour + 2-opt

In [ ]:
def solve_tsp(normals: list, start, end, costs: dict) -> list:
    """Nearest-neighbour construction followed by 2-opt improvement."""
    def get_c(a, b):
        return min(costs.get(f"{a}|{b}", math.inf),
                   costs.get(f"{b}|{a}", math.inf))

    if not normals:
        return [c for c in [start, end] if c is not None]

    to_visit = list(normals)
    route = []

    # --- construction ---
    if start:
        route.append(start)
        current = start
    else:
        best = min(to_visit, key=lambda c: sum(get_c(c, o) for o in to_visit if o != c) / max(len(to_visit)-1, 1))
        to_visit.remove(best)
        route.append(best)
        current = best

    while to_visit:
        nn = min(to_visit, key=lambda c: get_c(current, c))
        to_visit.remove(nn)
        route.append(nn)
        current = nn

    if end and route[-1] != end:
        route.append(end)

    # --- 2-opt ---
    sf = 1 if start else 0
    ef = 1 if end   else 0
    def tour_cost(r): return sum(get_c(r[i], r[i+1]) for i in range(len(r)-1))

    for _ in range(20):
        improved = False
        for i in range(sf, len(route) - 1 - ef):
            for j in range(i + 1, len(route) - ef):
                nr = route[:i+1] + route[i+1:j+1][::-1] + route[j+1:]
                if tour_cost(nr) < tour_cost(route) - 0.001:
                    route = nr
                    improved = True
        if not improved:
            break

    return route

print("solve_tsp defined")

---

## 7. Sample — Small-Scale Demonstration

We build a minimal graph with 5 cities and 6 edges, run Dijkstra to get all-pairs shortest paths, then solve a TSP tour.

```
    A ---10--- B
    |  \       |
    4    15    7
    |      \   |
    C ---9-- D-3-E
```

In [ ]:
# --- Build sample graph ---

nodes = ["A", "B", "C", "D", "E"]

raw_edges = [
    # (id, nodeA, nodeB, distance_km, speed_kmh, traffic)
    ("e1", "A", "B", 10.0, 90,  0.1),
    ("e2", "A", "C",  4.0, 50,  0.3),
    ("e3", "A", "D", 15.0, 110, 0.0),
    ("e4", "B", "D",  7.0, 90,  0.5),
    ("e5", "C", "D",  9.0, 70,  0.2),
    ("e6", "D", "E",  3.0, 50,  0.0),
]

edges = [
    Edge(eid, a, b, [Segment(d, spd, traf)], d)
    for eid, a, b, d, spd, traf in raw_edges
]

adj = build_adj(edges, nodes)

print("Graph built — adjacency list:")
for node, neighbors in adj.items():
    for e in neighbors:
        print(f"  {e.frm} -> {e.to}  dist={e.dist:.1f} km  time={e.time:.2f} min  fuel={e.fuel:.3f} L")

In [ ]:
# --- All-pairs shortest paths (by time) ---

cities = ["A", "B", "C", "D", "E"]
pair_paths = {}
pair_costs = {}

for src in cities:
    c, prev, prev_edge = dijkstra(adj, src, "time")
    for tgt in cities:
        if tgt == src:
            continue
        p = get_path(c, prev, prev_edge, src, tgt)
        pair_paths[f"{src}|{tgt}"] = p
        pair_costs[f"{src}|{tgt}"] = p["cost"]

print("All-pairs shortest paths (time in minutes):")
print(f"{'':4}", end="")
for tgt in cities:
    print(f"{tgt:>8}", end="")
print()
for src in cities:
    print(f"{src:4}", end="")
    for tgt in cities:
        if src == tgt:
            print(f"{'—':>8}", end="")
        else:
            v = pair_costs.get(f"{src}|{tgt}", math.inf)
            print(f"{v:>8.2f}", end="")
    print()

In [ ]:
# --- TSP tour: visit B, C, E — departing from A ---

tour = solve_tsp(["B", "C", "E"], start="A", end=None, costs=pair_costs)

tour_cost = sum(
    min(pair_costs.get(f"{tour[i]}|{tour[i+1]}", math.inf),
        pair_costs.get(f"{tour[i+1]}|{tour[i]}", math.inf))
    for i in range(len(tour) - 1)
)

print("Optimal tour (nearest-neighbour + 2-opt):")
print(" -> ".join(tour))
print(f"Total travel time: {tour_cost:.2f} min")

print("\nPer-leg breakdown:")
for i in range(len(tour) - 1):
    frm, to = tour[i], tour[i+1]
    leg_cost = pair_costs.get(f"{frm}|{to}", pair_costs.get(f"{to}|{frm}", math.inf))
    path_nodes = pair_paths.get(f"{frm}|{to}", {}).get("nodes", [frm, to])
    print(f"  {frm} -> {to}  via {' -> '.join(path_nodes)}  ({leg_cost:.2f} min)")

In [ ]:
# --- Compare all permutations (brute-force, feasible only for small n) ---

from itertools import permutations

def perm_cost(perm, start, costs):
    route = ([start] if start else []) + list(perm)
    return sum(
        min(costs.get(f"{route[i]}|{route[i+1]}", math.inf),
            costs.get(f"{route[i+1]}|{route[i]}", math.inf))
        for i in range(len(route) - 1)
    )

to_visit = ["B", "C", "E"]
best_perm, best_cost = None, math.inf
for perm in permutations(to_visit):
    c = perm_cost(perm, "A", pair_costs)
    if c < best_cost:
        best_cost = c
        best_perm = perm

optimal_route = ["A"] + list(best_perm)
print("Brute-force optimal:", " -> ".join(optimal_route), f"({best_cost:.2f} min)")
print("Heuristic tour:     ", " -> ".join(tour),           f"({tour_cost:.2f} min)")
gap = (tour_cost - best_cost) / best_cost * 100 if best_cost > 0 else 0
print(f"Optimality gap: {gap:.1f}%")

---

## 8. Cost Formulas Summary

| Metric | Formula | Unit |
|--------|---------|------|
| Traffic efficiency | $\text{eff} = \max(0.05,\ 1 - 0.6t)$ | — |
| Effective speed | $v^{\text{eff}} = \min(v, v^{\text{max}}) \cdot \text{eff}$ | km/h |
| Segment time | $T = \dfrac{d}{v^{\text{eff}}} \times 60$ | min |
| Segment fuel | $F = d \cdot (0.07 + 0.05t)$ | L |
| Route total time | $T_{\text{route}} = \displaystyle\sum_{s} T_s$ | min |
| Route total distance | $D_{\text{route}} = \displaystyle\sum_{s} d_s$ | km |
| Route total fuel | $F_{\text{route}} = \displaystyle\sum_{s} F_s$ | L |
| Fleet score (time) | $S = \max_v T_v^{\text{route}}$ | min |
| Fleet score (distance/fuel) | $S = \displaystyle\sum_v X_v^{\text{route}}$ | km or L |

> The **time score** uses the maximum across vehicles (the slowest vehicle determines the total duration).  
> The **distance and fuel scores** use the sum across all vehicles.

---

## 9. Constraint Checking

After route computation, each vehicle route is checked against optional constraints:

```python
if obj.maxTime     is not None and route_time     > obj.maxTime:     breach
if obj.maxDistance is not None and route_distance > obj.maxDistance: breach
if obj.totalUnits  > 0         and units_carried  > vehicle.capacity: breach
```

A route is marked **feasible** if and only if it has zero breaches.  
The global result is feasible only if all vehicle routes are feasible.